# Literature Review — Technology Transfer in Logistics

**Research question:** What does the academic literature say about technology transfer and adoption in logistics / supply chain, and how does it inform the XaaS thesis (digital readiness gaps explaining LT SME export under-representation)?

**Sources:** OpenAlex (open), Semantic Scholar (open), CORE (free key), Scopus + ScienceDirect (Elsevier key — currently stubbed; results merge in automatically once `ELSEVIER_API_KEY` is set in `.env`).

**Phased approach:**
1. **Phase 1 (this notebook, today):** broad pull → `lit_raw.csv` with abstracts + metadata.
2. **Phase 2 (manual):** shortlist top 20-30 by relevance.
3. **Phase 3 (later cell):** LLM-extract structured fields (research question, method, finding, applicability) → `lit_review.csv`.

This notebook is **self-contained**: re-running from a clean kernel reproduces the CSV from scratch.


In [ ]:
import os
import sys
from pathlib import Path

# Make `lit` package importable. Notebook runs from research/notebooks/, so add research/ to path.
sys.path.insert(0, str(Path('..').resolve()))

# Load .env if present (so OPENALEX_EMAIL, CORE_API_KEY, ELSEVIER_API_KEY land in os.environ).
env_path = Path('..').resolve() / '.env'
if env_path.exists():
    for line in env_path.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith('#') and '=' in line:
            k, v = line.split('=', 1)
            if v and k not in os.environ:
                os.environ[k] = v
    print(f'Loaded .env from {env_path}')
else:
    print(f'No .env at {env_path} (use .env.example as a template).')

# Show which credentials are present (without printing the values).
for k in ['OPENALEX_EMAIL', 'CORE_API_KEY', 'SEMANTIC_SCHOLAR_API_KEY', 'ELSEVIER_API_KEY']:
    print(f'  {k}: {"set" if os.environ.get(k) else "unset"}')


## 1. Define the search query

In [ ]:
# Broad query: technology transfer / adoption in logistics or supply chain.
# Phase 2 will narrow this; Phase 1 wants a wide net.
QUERY      = '"technology transfer" OR "technology adoption" AND (logistics OR "supply chain")'
YEAR_FROM  = 2015
YEAR_TO    = 2025
MAX_PER_SRC = 200   # raise to 500 once you've validated coverage on a small pull

print(f"Query: {QUERY}")
print(f"Years: {YEAR_FROM}-{YEAR_TO}")
print(f"Cap per source: {MAX_PER_SRC}")


## 2. Run the pipeline (all sources, deduped)

In [ ]:
from lit.pipeline import run_search, save_csv

df, stats = run_search(
    query=QUERY,
    year_from=YEAR_FROM,
    year_to=YEAR_TO,
    max_per_source=MAX_PER_SRC,
)

print('\nPer-source results:')
for s in stats:
    err = f'  ERROR: {s.error}' if s.error else ''
    print(f'  {s.name:<18} raw={s.raw_count:>4}   elapsed={s.elapsed_s}s{err}')

print(f'\nDeduped total: {len(df)} unique papers')
print(f'With abstract:  {(df["abstract"].astype(bool)).sum()}')
print(f'With DOI:       {df["doi"].notna().sum()}')


## 3. Save raw literature CSV

In [ ]:
OUT_PATH = Path('..') / 'data' / 'literature' / 'lit_raw.csv'
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
save_csv(df, OUT_PATH)
print(f'Saved {len(df)} rows -> {OUT_PATH.resolve()}')


## 4. Coverage diagnostics

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# Papers per source (after dedupe — combined sources show as 'a+b')
src_counts = df['source'].value_counts()
axes[0].barh(src_counts.index, src_counts.values, color='#3498db')
axes[0].set_title('Papers per source (deduped)')
axes[0].set_xlabel('Count')
for i, v in enumerate(src_counts.values):
    axes[0].text(v + 0.5, i, str(v), va='center', fontsize=9)

# Year distribution
yr = df['year'].dropna().astype(int)
axes[1].hist(yr, bins=range(YEAR_FROM, YEAR_TO + 2), color='#2ecc71', edgecolor='white')
axes[1].set_title('Publication year distribution')
axes[1].set_xlabel('Year')

# Citation count distribution (log scale, drop NaNs and zeros for visibility)
cits = df['citation_count'].dropna()
cits_pos = cits[cits > 0]
if len(cits_pos):
    axes[2].hist(cits_pos, bins=30, color='#e67e22', edgecolor='white')
    axes[2].set_yscale('log')
    axes[2].set_title('Citation count distribution (log y)')
    axes[2].set_xlabel('Citations')
else:
    axes[2].text(0.5, 0.5, 'No citation data available',
                 ha='center', va='center', transform=axes[2].transAxes)

plt.tight_layout()
plt.savefig('../reports/lit_coverage_overview.png', dpi=150)
plt.show()


## 5. Top-20 most-cited papers in the pull

In [ ]:
top = df.dropna(subset=['citation_count']).head(20)[
    ['title', 'authors', 'year', 'venue', 'citation_count', 'source', 'doi']
].copy()
top['authors'] = top['authors'].apply(lambda xs: '; '.join(xs[:3]) + ('; ...' if len(xs) > 3 else ''))
top['title'] = top['title'].str.slice(0, 90)
top.reset_index(drop=True)


## 6. Phase 3 — LLM structured extraction (placeholder)

Run this **after** you've shortlisted ~20-30 papers from `lit_raw.csv` (mark them in a `shortlist` column, or filter by min citation count + manual review).

The cell below is a stub. Uncomment and run when ready. Cost estimate: ~30 papers × ~2k tokens out per paper ≈ $0.05-0.10 with Haiku 4.5, $0.50-1 with Sonnet 4.6. Use Haiku for first pass; Sonnet only if extraction quality is too low.

```python
# from anthropic import Anthropic
# import json
#
# client = Anthropic()
# shortlist = pd.read_csv('../data/literature/lit_raw.csv')  # or your filtered subset
# shortlist = shortlist[shortlist['shortlist'] == True].copy()
#
# def extract(paper_row):
#     prompt = f"""Extract structured fields from this paper abstract.
#
# Title: {paper_row['title']}
# Year: {paper_row['year']}
# Abstract: {paper_row['abstract']}
#
# Return JSON with keys: research_question, method, key_finding, sample, applicability_to_LT_SME_exports.
# Each value 1-2 sentences. If a field is not derivable from the abstract, return null for that key."""
#     msg = client.messages.create(
#         model='claude-haiku-4-5-20251001',
#         max_tokens=600,
#         messages=[{'role': 'user', 'content': prompt}],
#     )
#     return json.loads(msg.content[0].text)
#
# extracted = shortlist.apply(extract, axis=1, result_type='expand')
# review = pd.concat([shortlist[['title', 'year', 'doi', 'source']], extracted], axis=1)
# review.to_csv('../data/literature/lit_review.csv', index=False)
# print(f'Wrote {len(review)} structured extractions.')
```


## 7. Integration with existing notebooks

Once `lit_review.csv` exists, downstream notebooks can join evidence to charts:

```python
lit = pd.read_csv('../data/literature/lit_review.csv')
# e.g. cite top finding next to the digital-readiness chart in nb04
relevant = lit[lit['key_finding'].str.contains('digital', case=False, na=False)]
```

The CSV is the single source of truth; chart annotations stay shallow (just cite DOI), and the deck pulls full quotes from `lit_review.csv`.